In [0]:
# ============================================================================
# SUPPLEMENTARY TABLE  (STEP 1 of 2) — per-event enrichment
#
# Goal: a NEW table built on top of tw_deviation_data_formatted_rdq. That source
# is at DATE/ROW grain (many rows per Event_Number), so the final table keeps
# that grain and the per-event enrichment repeats across an event's rows
# (duplicates are EXPECTED). This cell assembles the per-event enrichment
# (one row per pr_id = Event_Number); the next cell joins it onto the source.
#
# Enrichment pulled from deviation_embed_input (one row per pr_id):
#   - deterministic entities  → parsed out of deterministic_context into typed
#                               arrays (clinical_ids, documents/SOPs, acronyms,
#                               cros, devices, vendors)
#   - llm_context             → LLM-generated situating paragraph (retrieved ctx)
#   - deterministic_context   → raw resolved-references blob (kept for reference)
#
# Sources:
#   - us_gmsgq_dev.gms_us_alyt.deviation_embed_input               (enrichment)
#   - us_gmsgq_dev.gms_us_mart.tw_deviation_data_formatted_rdq     (source rows)
#   - topic assignment table (TBD — a topic model has not been chosen yet)
# ============================================================================
import re
import json
import pandas as pd
import numpy as np
from pyspark.sql import functions as F, types as T

CATALOG = "us_gmsgq_dev"
ALYT    = "gms_us_alyt"
MART    = "gms_us_mart"

EMBED_INPUT  = f"{CATALOG}.{ALYT}.deviation_embed_input"
SOURCE_TABLE = f"{CATALOG}.{MART}.tw_deviation_data_formatted_rdq"
OUTPUT_TABLE = f"{CATALOG}.{ALYT}.deviation_supplementary"

# Column on SOURCE_TABLE that identifies the event (cast to string == pr_id).
SRC_ID_COL   = "Event_Number"

# Topic assignments (pr_id, topic) — populated once a topic model is selected
# from topic_modeling.ipynb (BERTopic / Top2Vec / NMF / LDA). Left null for now.
TOPIC_TABLE  = f"{CATALOG}.{ALYT}.deviation_topic_assignments"


# ---- Parse deterministic_context into structured entity arrays -------------
# deterministic_context is a blob of resolved-reference blocks joined by "---",
# each block starting with:  [TYPE] "key"\nCanonical: <name>\n<enrichment...>
def parse_deterministic_context(ctx: str) -> dict:
    """Parse the deterministic_context blob into entity-type arrays."""
    result = {
        "clinical_ids": [],
        "documents": [],
        "acronyms": [],
        "cros": [],
        "devices": [],
        "vendors": [],
    }
    if not ctx:
        return result

    for block in ctx.split("---"):
        block = block.strip()
        if not block:
            continue
        # First line of each block: [TYPE] "key"
        m = re.match(r'\[(\w+)\]\s*"([^"]+)"', block)
        if not m:
            continue
        etype = m.group(1).upper()
        key = m.group(2)

        # Prefer the resolved canonical name when present
        canon_match = re.search(r'Canonical:\s*(.+)', block)
        canonical = canon_match.group(1).strip() if canon_match else key

        if etype == "CLINICAL_ID":
            result["clinical_ids"].append(canonical)
        elif etype == "DOCUMENT":
            result["documents"].append(canonical)
        elif etype == "ACRONYM":
            result["acronyms"].append(canonical)
        elif etype == "CRO":
            result["cros"].append(canonical)
        elif etype == "DEVICE":
            result["devices"].append(canonical)
        elif etype == "VENDOR":
            result["vendors"].append(canonical)

    # De-duplicate while preserving order
    return {k: list(dict.fromkeys(v)) for k, v in result.items()}


# ---- Load per-event enrichment and parse ----------------------------------
embed_pdf = (
    spark.table(EMBED_INPUT)
    .select("pr_id", "deterministic_context", "llm_context")
    .toPandas()
)
embed_pdf["pr_id"] = embed_pdf["pr_id"].astype(str)
embed_pdf["deterministic_context"] = embed_pdf["deterministic_context"].fillna("")
embed_pdf["llm_context"] = embed_pdf["llm_context"].fillna("")

parsed = embed_pdf["deterministic_context"].apply(parse_deterministic_context)

entity_df = pd.DataFrame({
    "pr_id":                 embed_pdf["pr_id"],
    "clinical_ids":          parsed.apply(lambda x: x["clinical_ids"]),
    "documents":             parsed.apply(lambda x: x["documents"]),
    "acronyms":              parsed.apply(lambda x: x["acronyms"]),
    "cros":                  parsed.apply(lambda x: x["cros"]),
    "devices":               parsed.apply(lambda x: x["devices"]),
    "vendors":               parsed.apply(lambda x: x["vendors"]),
    "llm_context":           embed_pdf["llm_context"],
    "deterministic_context": embed_pdf["deterministic_context"],
})

print(f"Parsed {len(entity_df):,} events")
print(f"  with clinical_ids: {(entity_df['clinical_ids'].str.len() > 0).sum()}")
print(f"  with documents:    {(entity_df['documents'].str.len() > 0).sum()}")
print(f"  with acronyms:     {(entity_df['acronyms'].str.len() > 0).sum()}")
print(f"  with cros:         {(entity_df['cros'].str.len() > 0).sum()}")
print(f"  with devices:      {(entity_df['devices'].str.len() > 0).sum()}")
print(f"  with vendors:      {(entity_df['vendors'].str.len() > 0).sum()}")
print(f"  with llm_context:  {(entity_df['llm_context'].str.len() > 0).sum()}")

In [ ]:
# ============================================================================
# SUPPLEMENTARY TABLE  (STEP 2 of 2) — join enrichment onto the source rows
#
# The source table is at DATE/ROW grain (many rows per Event_Number). We keep
# every source row and LEFT-join the per-event enrichment on Event_Number, so
# the entities / llm_context / topic repeat across an event's rows. This 1-to-
# many join is what produces the expected duplicates.
# ============================================================================
from pyspark.sql.utils import AnalysisException

# ---- per-event enrichment (pandas -> Spark, with explicit array schema) ----
enrich_schema = T.StructType([
    T.StructField("pr_id",                 T.StringType(),               False),
    T.StructField("clinical_ids",          T.ArrayType(T.StringType()),  True),
    T.StructField("documents",             T.ArrayType(T.StringType()),  True),
    T.StructField("acronyms",              T.ArrayType(T.StringType()),  True),
    T.StructField("cros",                  T.ArrayType(T.StringType()),  True),
    T.StructField("devices",               T.ArrayType(T.StringType()),  True),
    T.StructField("vendors",               T.ArrayType(T.StringType()),  True),
    T.StructField("llm_context",           T.StringType(),               True),
    T.StructField("deterministic_context", T.StringType(),               True),
])

# Build from plain Python tuples so Spark reads the list columns as arrays.
enrich_rows = [
    (r.pr_id, list(r.clinical_ids), list(r.documents), list(r.acronyms),
     list(r.cros), list(r.devices), list(r.vendors),
     r.llm_context, r.deterministic_context)
    for r in entity_df.itertuples(index=False)
]
enrich_sdf = spark.createDataFrame(enrich_rows, schema=enrich_schema)

# ---- source rows (date/row grain) keyed by Event_Number == pr_id ----------
src_keyed = (
    spark.table(SOURCE_TABLE)
    .withColumn("pr_id", F.col(f"`{SRC_ID_COL}`").cast("string"))
)

# ---- LEFT join: keep all source rows, fan enrichment across each event -----
supp = src_keyed.join(enrich_sdf, on="pr_id", how="left")

# ---- topic (from a topic model — not chosen yet; join if the table exists) --
try:
    topic_sdf = (
        spark.table(TOPIC_TABLE)
        .select(
            F.col("pr_id").cast("string").alias("pr_id"),
            F.col("topic").cast("string").alias("topic"),
        )
    )
    supp = supp.join(topic_sdf, on="pr_id", how="left")
    print(f"Joined topic assignments from {TOPIC_TABLE}")
except AnalysisException:
    supp = supp.withColumn("topic", F.lit(None).cast("string"))
    print(f"(!) {TOPIC_TABLE} not found — 'topic' left null until a topic model "
          f"is selected in topic_modeling.ipynb")

# Drop the join helper; keep the original Event_Number column.
supp = supp.drop("pr_id")

# ---- Write the supplementary table ----------------------------------------
(supp.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE))

n_rows   = supp.count()
n_events = supp.select(f"`{SRC_ID_COL}`").distinct().count()
print(f"✓ {OUTPUT_TABLE}")
print(f"  {n_rows:,} rows across {n_events:,} distinct {SRC_ID_COL} "
      f"(date/row grain — duplicates per event are expected)")
display(supp.limit(10))